<a href="https://colab.research.google.com/github/Shanmuganathan75/QM640-WALSH-CAPSTONE/blob/main/03f_merge_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# QM 640 Capstone — Step 3f: Merge Manual Review Back Into Master Worksheet

Takes your completed `screening_TO_REVIEW.csv` (with `is_genuine_ai_event`
and `announcement_type` filled in by hand) and merges those two columns
back into the master `screening_worksheet.csv`, matched by `accession_no`.

**Run this after `03e_fetch_snippets.ipynb` and after you've finished filling in `screening_TO_REVIEW.csv` and
pushed your updated version back to the repo** (re-upload the completed
CSV into `data/raw/screening_TO_REVIEW.csv`, replacing the blank one,
before running this notebook).

## Cell 1 — Clone (or pull) the repo

Run this first, every session. `BASE_DIR` is the single fixed path every
other cell in this notebook reads from and writes to.

In [ ]:
import os
from google.colab import userdata

GITHUB_USERNAME = "Shanmuganathan75"      # edit if different
REPO_NAME = "QM640-WALSH-CAPSTONE"        # must match the repo URL EXACTLY (hyphens included)
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')  # set once via Colab Secrets (key icon, left sidebar)

BASE_DIR = f"/content/{REPO_NAME}"
remote_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    # Wipe any stale/incomplete folder from a previous failed attempt before retrying
    !rm -rf {BASE_DIR}
    !git clone {remote_url} {BASE_DIR}
else:
    !git -C {BASE_DIR} pull

# Fail loudly instead of silently continuing with a non-git folder -
# this is exactly the bug that caused "fatal: not a git repository" earlier.
if not os.path.exists(os.path.join(BASE_DIR, ".git")):
    raise RuntimeError(
        f"Clone failed: no .git folder found at {BASE_DIR}.\n"
        f"Check that GITHUB_USERNAME (\'{GITHUB_USERNAME}\') and REPO_NAME "
        f"(\'{REPO_NAME}\') exactly match your repo URL (case and hyphens "
        f"included), and that GITHUB_TOKEN is set in Colab Secrets with "
        f"Contents: Read and write access."
    )

for sub in ["scripts", "data/raw", "data/processed/returns", "results"]:
    os.makedirs(os.path.join(BASE_DIR, sub), exist_ok=True)

!git -C {BASE_DIR} config user.email "Shan_muganathan@yahoo.com"
!git -C {BASE_DIR} config user.name "Shanmuganathan Ekambaram"

print("Repo ready at:", BASE_DIR)

Cloning into '/content/QM640-WALSH-CAPSTONE'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (33/33), done.
remote: Total 91 (delta 33), reused 72 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 819.70 KiB | 4.79 MiB/s, done.
Resolving deltas: 100% (33/33), done.
Repo ready at: /content/QM640-WALSH-CAPSTONE


## Cell 2 — Install dependencies

In [ ]:
!pip install -q pandas

## Cell 3 — Configuration

In [ ]:
import os

RAW_DIR = os.path.join(BASE_DIR, "data/raw")
SCREENING_FILE = os.path.join(RAW_DIR, "screening_worksheet.csv")
REVIEW_FILE = os.path.join(RAW_DIR, "screening_TO_REVIEW.csv")

## Cell 4 — Load both files and check completeness

In [ ]:
import pandas as pd

master = pd.read_csv(SCREENING_FILE)
review = pd.read_csv(REVIEW_FILE)

print(f"Master worksheet: {len(master)} rows")
print(f"Review file: {len(review)} rows")

filled = review["is_genuine_ai_event"].notna() & (review["is_genuine_ai_event"].astype(str).str.strip() != "")
print(f"\nRows with is_genuine_ai_event filled in: {filled.sum()} / {len(review)}")
if filled.sum() < len(review):
    print(f"WARNING: {len(review) - filled.sum()} rows still blank. "
          f"You can still merge partial progress, but these rows won\'t "
          f"count toward your reviewed total yet.")

Master worksheet: 3092 rows
Review file: 245 rows

Rows with is_genuine_ai_event filled in: 245 / 245


## Cell 5 — Merge classifications back into the master worksheet

In [ ]:
update_cols = ["accession_no", "is_genuine_ai_event", "announcement_type"]

# 1. Prepare the `review` DataFrame for merging.
#    Ensure 'accession_no' in `review` is unique before using it for updates.
#    If there are duplicates in `review['accession_no']`, we'll keep the last entry.
review_updates_df = review[update_cols].drop_duplicates(subset=['accession_no'], keep='last')

# 2. Reset the index of `master` temporarily to treat 'accession_no' as a regular column.
#    `master` is currently indexed by accession_no from the previous run, so reset it.
master_temp = master.reset_index()

# 3. Perform a left merge to bring the updated columns from `review_updates_df` into `master_temp`.
#    This will add columns like 'is_genuine_ai_event_review' and 'announcement_type_review'.
merged_df = pd.merge(master_temp, review_updates_df, on='accession_no', how='left', suffixes=('', '_review'))

# 4. Update the original 'is_genuine_ai_event' and 'announcement_type' columns in `master_temp`.
#    The `fillna` method here ensures that if a value exists in the '_review' column (meaning it was
#    present in the `review_updates_df`), it will be used; otherwise, the original value from `master_temp`
#    is retained. This replicates the behavior of `combine_first` where the 'updates' take precedence.
master_temp['is_genuine_ai_event'] = merged_df['is_genuine_ai_event_review'].fillna(master_temp['is_genuine_ai_event'])
master_temp['announcement_type'] = merged_df['announcement_type_review'].fillna(master_temp['announcement_type'])

# 5. Assign the updated DataFrame back to `master`.
master = master_temp

master.to_csv(SCREENING_FILE, index=False)
print(f"Merged and saved -> {SCREENING_FILE}")

confirmed = master["is_genuine_ai_event"].astype(str).str.upper() == "Y"
print(f"\nConfirmed genuine AI events so far: {confirmed.sum()}")
print(master.loc[confirmed, "announcement_type"].value_counts())

Merged and saved -> /content/QM640-WALSH-CAPSTONE/data/raw/screening_worksheet.csv

Confirmed genuine AI events so far: 71
announcement_type
M&A            36
R&D            21
partnership    14
Name: count, dtype: int64


## Commit and push results back to GitHub

In [ ]:
!git -C {BASE_DIR} add "data/raw/screening_worksheet.csv"
!git -C {BASE_DIR} commit -m "Step 3f: merge manual classifications into master worksheet"
!git -C {BASE_DIR} push

[main 034a455] Step 3f: merge manual classifications into master worksheet
 1 file changed, 3093 insertions(+), 3093 deletions(-)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (5/5), 46.16 KiB | 1.71 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/Shanmuganathan75/QM640-WALSH-CAPSTONE.git
   3fb7591..034a455  main -> main


## Sanity check against your N=159 target

In [ ]:
print(f"Confirmed events: {confirmed.sum()}")
print(f"Target minimum (RQ2 bottleneck): 159")
print(f"Practical collection goal: 175-190")

if confirmed.sum() < 159:
    print("\nStill below target - keep reviewing remaining rows in "
          "screening_TO_REVIEW.csv, or widen Step 1 further.")
elif confirmed.sum() < 175:
    print("\nAbove minimum but below practical target - fine to proceed, "
          "or keep reviewing remaining rows for more buffer.")
else:
    print("\nWithin practical target range - ready to move toward the "
          "kappa check once your independent reviewer is also done.")

Confirmed events: 71
Target minimum (RQ2 bottleneck): 159
Practical collection goal: 175-190

Still below target - keep reviewing remaining rows in screening_TO_REVIEW.csv, or widen Step 1 further.
